In [1]:
import json, re, itertools
from collections import Counter
import pandas as pd

PATH = "market_data_2026-11-15.json"  # ~2.9 GB, one flat JSON array of market records

_WS = re.compile(r"[\s,]*")

def iter_records(path, chunk_size=1 << 22):
    """Stream records from a big top-level JSON array without loading the whole file."""
    dec = json.JSONDecoder()
    with open(path) as f:
        buf = f.read(chunk_size)
        pos = buf.index("[") + 1
        while True:
            pos = _WS.match(buf, pos).end()
            if pos < len(buf) and buf[pos] == "]":
                return
            try:
                obj, pos = dec.raw_decode(buf, pos)
                yield obj
            except json.JSONDecodeError:  # record cut off at chunk boundary
                more = f.read(chunk_size)
                if not more:
                    return
                buf = buf[pos:] + more
                pos = 0

# Load a sample (first N records) into a DataFrame
N = 200_000
df = pd.DataFrame(itertools.islice(iter_records(PATH), N))
for c in ["open_interest", "daily_volume", "block_volume"]:  # stored as strings
    df[c] = pd.to_numeric(df[c])
df.info()
df.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   date           200000 non-null  object 
 1   ticker_name    200000 non-null  object 
 2   report_ticker  200000 non-null  object 
 3   payout_type    200000 non-null  object 
 4   open_interest  200000 non-null  float64
 5   daily_volume   200000 non-null  float64
 6   block_volume   200000 non-null  float64
 7   high           200000 non-null  int64  
 8   low            200000 non-null  int64  
 9   status         200000 non-null  object 
dtypes: float64(3), int64(2), object(5)
memory usage: 15.3+ MB


,date,ticker_name,report_ticker,payout_type,open_interest,daily_volume,block_volume,high,low,status
0,2026-11-15,AMAZONFTC-29DEC31,AMAZONFTC,Binary Option,6849.96,7.00,0.0,49,49,active
1,2026-11-15,NYTOAI-27DEC31,NYTOAI,Binary Option,17119.37,0.00,0.0,33,33,active
2,2026-11-15,MOON-26DEC31,MOON,Binary Option,81175.00,86.00,0.0,0,0,active
3,2026-11-15,TESLAOPTIMUS-26DEC31,TESLAOPTIMUS,Binary Option,171954.46,344.31,0.0,4,2,active
4,2026-11-15,STARSHIPMARS-29DEC31,STARSHIPMARS,Binary Option,9965.71,135.84,0.0,15,12,active
5,2026-11-15,APPLEUS-29DEC31,APPLEUS,Binary Option,7667.05,0.00,0.0,21,21,active
6,2026-11-15,OAIAGI-29,OAIAGI,Binary Option,84157.35,15.00,0.0,67,67,active
7,2026-11-15,USCLIMATE-2025,USCLIMATE,Binary Option,6829.74,0.00,0.0,4,4,active
8,2026-11-15,USCLIMATE-2030,USCLIMATE,Binary Option,1753.65,25.00,0.0,16,16,active
9,2026-11-15,EVSHARE-30JAN-10,EVSHARE,Binary Option,541.10,0.00,0.0,84,84,active


In [2]:
# Quick look at the sample
print(df["status"].value_counts(), "\n")
print(df["payout_type"].value_counts(), "\n")
print("unique series (report_ticker):", df["report_ticker"].nunique())
print("unique markets (ticker_name):", df["ticker_name"].nunique(), "\n")
print("Top series by open interest:")
print(df.groupby("report_ticker")[["open_interest", "daily_volume"]].sum()
        .sort_values("open_interest", ascending=False).head(15))
df.describe()

status
active       102923
finalized     94214
closed         2863
Name: count, dtype: int64 

payout_type
Binary Option    200000
Name: count, dtype: int64 

unique series (report_ticker): 4403
unique markets (ticker_name): 200000 

Top series by open interest:
               open_interest  daily_volume
report_ticker                             
KXPRESNOMD      1.884893e+08    1141283.93
KXSB            1.805381e+08    4736283.18
KXPRESPERSON    6.645918e+07     435730.38
KXPRESNOMR      6.126524e+07     875927.22
KXNCAAF         5.897908e+07     905799.52
KXNBA           5.081273e+07     570827.23
KXNFLMVP        3.612885e+07    1500932.58
SENATEAK        3.496128e+07    2753757.87
KXHEISMAN       3.177954e+07     454108.12
KXMARMAD        1.909931e+07     758027.45
KXNFLMATCHUP    1.536302e+07     779841.68
KXNFLNFCCHAMP   1.298529e+07     211170.54
KXBTCY          1.278893e+07     170534.63
KXNFLAFCCHAMP   1.201599e+07     148019.56
KXALIENS        1.148248e+07     152410.09


,open_interest,daily_volume,block_volume,high,low
count,2.000000e+05,2.000000e+05,200000.0,200000.000000,200000.000000
mean,5.916608e+03,4.399890e+02,0.0,47.419225,47.245470
std,1.402092e+05,4.844216e+04,0.0,14.565622,14.642923
min,0.000000e+00,0.000000e+00,0.0,0.000000,0.000000
25%,0.000000e+00,0.000000e+00,0.0,50.000000,50.000000
50%,0.000000e+00,0.000000e+00,0.0,50.000000,50.000000
75%,0.000000e+00,0.000000e+00,0.0,50.000000,50.000000
max,2.745357e+07,1.464965e+07,0.0,99.000000,99.000000


In [3]:
# OPTIONAL: one streaming pass over the FULL file (~10M rows, ~30-60s) - counts only, low memory
total = 0
status_counts, payout_counts, series_oi = Counter(), Counter(), Counter()
for r in iter_records(PATH):
    total += 1
    status_counts[r["status"]] += 1
    payout_counts[r["payout_type"]] += 1
    series_oi[r["report_ticker"]] += float(r["open_interest"])
print("total records:", total)
print("status:", dict(status_counts))
print("payout_type:", dict(payout_counts))
print("unique series:", len(series_oi))
pd.Series(series_oi).sort_values(ascending=False).head(20)

total records: 9575204
status: {'active': 346965, 'finalized': 9223519, 'closed': 4661, 'determined': 59}
payout_type: {'Binary Option': 9575204}
unique series: 7602


KXMVECROSSCATEGORY     2.900997e+08
KXPRESNOMD             1.951470e+08
KXSB                   1.805381e+08
KXPRESPERSON           7.403919e+07
KXPRESNOMR             6.439831e+07
KXNCAAF                5.897908e+07
KXNBA                  5.081273e+07
KXNFLMVP               3.612885e+07
SENATEAK               3.496128e+07
KXHEISMAN              3.349916e+07
KXPGATOUR              2.812406e+07
KXMVECROSSCATEGORY0    2.403685e+07
KXMARMAD               1.921235e+07
KXNFLMATCHUP           1.536302e+07
KXNFLNFCCHAMP          1.298529e+07
KXBTCY                 1.278893e+07
KXNFLAFCCHAMP          1.201599e+07
KXALIENS               1.148248e+07
KXMIDTERMMOV           1.008334e+07
KXLLM1                 9.973049e+06
dtype: float64